# Module 3: LangChain Expression Language (LCEL) Deep Dive

In this notebook, we will explore:
1. **Simple Chain**: Building a basic `prompt | model | parser` pipeline.
2. **Runnable Protocol**: Exploring the difference between `invoke`, `batch`, and `stream`.
3. **RunnablePassthrough**: Forwarding inputs and assigning dictionary elements dynamically.
4. **RunnableParallel**: Running multiple operations concurrently.
5. **RunnableLambda**: Integrating custom Python functions into LCEL.

### Step 1: Initialize Chat Model Connection

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="google/gemma-2-9b-it:free",
    temperature=0.3,
)
print("Model client connected!")

---
## 1. The Simple Chain & Unified Methods

Let's build a chain using the pipe operator (`|`). We'll pipe a `ChatPromptTemplate` into our `ChatOpenAI` model, and finish with a `StrOutputParser` to clean up the message payload and yield a clean string output.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Write a 1-sentence slogan for a company that sells {product}.")
parser = StrOutputParser()

# Construct the LCEL chain
simple_chain = prompt | model | parser

print("Chain type:", type(simple_chain))

### Executing `invoke`, `batch`, and `stream` on our chain
Because `simple_chain` inherits from the base `Runnable` interface, we have access to consistent execution functions.

In [ ]:
# A. invoke() - single run
print("--- INVOKE RESULT ---")
print(simple_chain.invoke({"product": "organic solar panels"}))

# B. batch() - run multiple items concurrently via multi-threading
print("\n--- BATCH RESULT ---")
inputs = [
    {"product": "electric unicycles"},
    {"product": "biodegradable coffee cups"},
    {"product": "ergonomic coding keyboards"}
]
batch_results = simple_chain.batch(inputs)
for product_input, result in zip(inputs, batch_results):
    print(f"Product: {product_input['product']} => Slogan: {result.strip()}")

# C. stream() - receive chunks incrementally
print("\n--- STREAM RESULT ---")
for chunk in simple_chain.stream({"product": "flying cars"}):
    print(chunk, end="", flush=True)
print()

---
## 2. Managing Inputs with `RunnablePassthrough` & `RunnableParallel`

In realistic applications, our chain needs to receive an input dictionary, perform multiple operations (sometimes in parallel), and format downstream parameters.

Let's see how `RunnablePassthrough` can construct dictionaries dynamically.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Let's say we have an input string, but our prompt expects a dictionary with a key: {topic}
# We can use RunnablePassthrough to feed the input to the key dynamically:
input_chain = {"topic": RunnablePassthrough()} | prompt | model | parser

print(input_chain.invoke("smart dog collars"))

Now let's use `RunnablePassthrough.assign()` to add extra keys on-the-fly without destroying the original inputs.

In [ ]:
# This chain takes a dictionary, preserves the product name, and injects a generated description
description_prompt = ChatPromptTemplate.from_template("Write a 1-sentence product description for {product}.")
description_chain = description_prompt | model | parser

# We 'assign' a new key 'description' using our helper chain
enrichment_chain = RunnablePassthrough.assign(
    description=description_chain
)

input_data = {"product": "Self-cleaning cat litter boxes"}
enriched_data = enrichment_chain.invoke(input_data)

print("Enriched dictionary content:")
print(enriched_data)

### Running Tasks Concurrently with `RunnableParallel`

If we want to generate a product slogan and a product target audience analysis simultaneously, we can use `RunnableParallel` to execute both prompts concurrently.

In [ ]:
from langchain_core.runnables import RunnableParallel

slogan_prompt = ChatPromptTemplate.from_template("Write a catchy slogan for: {product}")
audience_prompt = ChatPromptTemplate.from_template("Describe the primary target audience for: {product} in one sentence.")

slogan_chain = slogan_prompt | model | parser
audience_chain = audience_prompt | model | parser

# Create parallel execution branches
parallel_runner = RunnableParallel(
    slogan=slogan_chain,
    target_audience=audience_chain
)

results = parallel_runner.invoke({"product": "Premium Matcha Whisk"})
print("Parallel Results Output:")
print(results)

---
## 3. Integrating Python Functions via `RunnableLambda`

Sometimes, you need to clean data, count tokens, log outputs, or query custom libraries inside your chain. We wrap these custom python functions with `RunnableLambda` to hook them directly into the pipe chain.

In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. Define standard python helper functions
def calculate_word_count(text: str) -> str:
    word_count = len(text.split())
    return f"{text} (Length: {word_count} words)"

def clean_formatting(text: str) -> str:
    # Remove double quotes or newlines
    return text.replace('"', '').strip()

# 2. Wrap them with RunnableLambda
word_counter = RunnableLambda(calculate_word_count)
cleaner = RunnableLambda(clean_formatting)

# 3. Wire them into the pipeline
processing_chain = slogan_prompt | model | parser | cleaner | word_counter

result = processing_chain.invoke({"product": "Noise-cancelling Sleep Earplugs"})
print("Processed result output:")
print(result)